In [ ]:
!pip install scikit-misc

In [ ]:
from skmisc.loess import loess
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
dados_cinematica = np.loadtxt('Dadoscorrida.txt')

In [ ]:
labels = ['ântero-posterior', 'médio-lateral', 'vertical']

plt.figure(figsize=(10,8))

for i in range(1,4):

    plt.subplot(3,1,i)

    plt.plot(dados_cinematica[:,0],
             dados_cinematica[:,i],
             'k-')

    plt.ylabel(f'{labels[i-1]} (m)')

plt.xlabel('Tempo (s)')

plt.tight_layout()
plt.show()

In [ ]:
#    Rotina para suavização de sinais cinemáticos utilizando RLOESS.
#   Esta rotina realiza a suavização de dados cinemáticos de corrida utilizando o método RLOESS.
#A função permite visualizar diferentes valores de janela de suavização (span), além de calcular
#os resíduos e as derivadas de velocidade e aceleração do sinal suavizado.

#Inicialmente, recomenda-se executar a rotina sem definir o parâmetro 'span', permitindo a
#visualização automática de diferentes tamanhos de janela para auxiliar na escolha do valor
#mais adequado para o sinal analisado.

#Exemplo:

#suavizar_cinematica(dados_cinematica, coluna=1,freq_aquis=300)

#Após a escolha da janela de suavização mais apropriada, a rotina deve ser executada novamente
#definindo o valor desejado para 'span' e habilitando a opção de salvamento dos resultados.

#Exemplo:

#suavizar_cinematica(dados_cinematica, coluna=1, span=0.22, salvar=True)

#A rotina gera gráficos contendo:
#- sinal original e suavizado;
#- resíduos;
#- derivada da velocidade;
#- derivada da aceleração.

#Quando salvar=True, a rotina exporta automaticamente um arquivo .txt contendo os dados
#processados.

#Rotina criada por Vitor Cimonetti.


def suavizar_cinematica(dados_cinematica, coluna=1, freq_aquis=300, span=None, salvar=False):


  if span is None:
    spans = np.arange(0.1, 0.95, 0.02)
  else:
    spans = [span]

  for i in spans:

    span_atual = i


    # Cria as características do Rloess
    span = i
    regressor = loess(dados_cinematica[:,0], dados_cinematica[:,coluna], span=span_atual, family='symmetric')
    regressor.fit()

    # Faz a predição em cima dos valores de x e gera os novos dados suavizados
    predicao = regressor.predict(dados_cinematica[:,0])
    Dados_corrida_suave = predicao.values

    residuo = dados_cinematica[:,coluna] - Dados_corrida_suave
    velocidade = (np.diff(Dados_corrida_suave))/(1/freq_aquis)
    aceleracao = (np.diff(velocidade))/(1/freq_aquis)

    fig, ax = plt.subplots(2, 2, figsize=(10, 6))

    # Sinal original e suavizado
    ax[0,0].plot(dados_cinematica[:,coluna], 'r-', linewidth=1, label='Original')
    ax[0,0].plot(Dados_corrida_suave, 'b-', linewidth=2, label='Suavizado')
    ax[0,0].set_title('Sinal original e suavizado')
    ax[0,0].set_ylabel('Coordenada (m)')
    ax[0,0].legend()

    # Resíduos
    ax[0,1].plot(residuo, '.', markersize=4)
    ax[0,1].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[0,1].set_title(f'Resíduos - janela = {span:.2f} (fração)')
    ax[0,1].set_ylabel('Resíduo (m)')

    # Velocidade
    ax[1,0].plot(velocidade, linewidth=1.5)
    ax[1,0].set_title('Derivada da velocidade')
    ax[1,0].set_xlabel('Frame')
    ax[1,0].set_ylabel('Velocidade (m/s)')

    # Aceleração
    ax[1,1].plot(aceleracao, linewidth=1.5)
    ax[1,1].set_title('Derivada da aceleração')
    ax[1,1].set_xlabel('Frame')
    ax[1,1].set_ylabel('Aceleração (m/s²)')

    plt.tight_layout()
    plt.show()

  if salvar == True:

        dados_saida = np.column_stack((dados_cinematica[:,0], dados_cinematica[:,coluna], Dados_corrida_suave, residuo))

        np.savetxt(
            f'dados_suavizados_eixo_{coluna}_span_{span:.2f}.txt',
            dados_saida,
            fmt='%.6f',
            delimiter='\t',
            header='Tempo\tOriginal\tSuavizado\tResiduo',
            comments=''
        )

  return Dados_corrida_suave, residuo, velocidade, aceleracao


In [ ]:
suavizar_cinematica(dados_cinematica, coluna=1, freq_aquis=300);

In [ ]:
suavizar_cinematica(dados_cinematica, coluna=1, span=0.22, salvar=True);